# DenMark in Colab

DenMark estimates, for each gene, the correlation rho between the local density of a cell population and the gene's expression per cell, from single-cell resolution spatial transcriptomics. This notebook runs the workflow of the [Get started in Python](https://stagill.github.io/DenMark/articles/python.html) page on the bundled toy data, a simulated tissue section with known answers: rho = 0.6 for geneA, 0 for geneB and -0.6 for geneC.

The first cell installs DenMark; the second builds CmdStan, which takes a few minutes the first time a Colab session runs it.

In [ ]:
%pip install -q "denmark[boundary,plot] @ git+https://github.com/StaGill/DenMark#subdirectory=python"

In [ ]:
import cmdstanpy
cmdstanpy.install_cmdstan(cores=2)

## The toy data and the tissue boundary

The 3,449 Tumor cells are the population analyzed; the 4,000 Other cells only delineate the tissue. The boundary is the alpha shape of all cells.

In [ ]:
import pandas as pd
import denmark as dm

toy = dm.datasets.load_toy()
tumor = toy[toy["celltype"] == "Tumor"].drop(columns="celltype")
truth = {"geneA": 0.6, "geneB": 0.0, "geneC": -0.6}
tissue = dm.alpha_shape(toy["x"], toy["y"])
gridded = dm.buildgridpp(10, tumor, gene="geneA", boundary=tissue)
fig = dm.plot_boundary(toy, tissue, grid=gridded, highlight=toy["celltype"] == "Tumor")

## Fit one gene

Short chains on a coarse grid keep this fast; real analyses use finer grids and the default sampler settings (four chains of 1,000 warm-up and 1,000 sampling iterations).

In [ ]:
fit = dm.DenMark(tumor, gene="geneA", boundary=tissue, grid_size=10, basis=10,
                 chains=2, iter_warmup=500, iter_sampling=500, seed=2024, quiet=True)
print(fit)
fig = dm.plot_fields(fit)

## Scan the genes and check against the truth

A gene is a positively (PCG) or negatively (NCG) density-correlated gene when the 95% credible interval of rho lies above or below 0.

In [ ]:
scan = dm.scan_genes(tumor, genes=["geneA", "geneB", "geneC"], boundary=tissue,
                     grid_size=10, basis=10, chains=2, iter_warmup=500,
                     iter_sampling=500, seed=2024, verbose=False)
scan["truth"] = scan["gene"].map(truth)
print(scan[["gene", "truth", "rho_mean", "rho_lower", "rho_upper", "class"]])
fig = dm.plot_scan(scan)

## Your own data

Upload a table with one row per cell (`x`, `y`, `celltype` and raw counts per gene), or start from an AnnData object, for example `sdata.tables["table"]` from spatialdata-io: `cells = dm.from_anndata(adata, celltype_key="celltype")`. Then compute the boundary from all cells and scan the cells of one population, as above. The [documentation](https://stagill.github.io/DenMark/) and the [FAQ](https://stagill.github.io/DenMark/articles/faq.html) cover the settings.